In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ✅ INSTALL DEPENDENCIES (Colab-compatible)
!pip install -q datasets transformers accelerate bitsandbytes faiss-cpu xformers peft torchvision pdfplumber sentence-transformers evaluate

# ✅ IMPORTS
from datasets import load_dataset
from PIL import Image
import torch
import numpy as np
import faiss
import random
from collections import Counter
import os
import json
import torch.nn.functional as F
import pdfplumber
import evaluate
from transformers import (
    BitsAndBytesConfig, AutoProcessor, LlavaForConditionalGeneration,
    SiglipProcessor, SiglipModel, BlipProcessor, BlipForConditionalGeneration
)
from sentence_transformers import SentenceTransformer

random.seed(42)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

from transformers import  BlipProcessor, BlipForConditionalGeneration

def js_divergence(p, q):
    p = torch.tensor(p, dtype=torch.float32)
    q = torch.tensor(q, dtype=torch.float32)
    p = F.softmax(p, dim=0)
    q = F.softmax(q, dim=0)
    m = 0.5 * (p + q)
    kl_pm = F.kl_div(m.log(), p, reduction='sum')
    kl_qm = F.kl_div(m.log(), q, reduction='sum')
    return 0.5 * (kl_pm + kl_qm).item()

class LightweightBLIPCaptioner:
    def __init__(self, model_name="Salesforce/blip-image-captioning-base", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = BlipProcessor.from_pretrained(model_name)
        self.model = BlipForConditionalGeneration.from_pretrained(model_name).to(self.device)

    def caption(self, image):
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=64)
        return self.processor.batch_decode(out, skip_special_tokens=True)[0]

class SigLIPRetriever:
    def __init__(self, model_name="google/siglip-base-patch16-224", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = SiglipProcessor.from_pretrained(model_name)
        self.model = SiglipModel.from_pretrained(model_name).to(self.device)
        self.text_data, self.image_data, self.image_captions = [], [], []
        self.embeddings, self.metadata = [], []
        self.index = None
        self.captioner = None

    def load_auto_captioner(self, model_name="Salesforce/blip-image-captioning-base"):
        self.captioner = LightweightBLIPCaptioner(model_name)

    def encode_texts(self, texts):
        inputs = self.processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(self.device)
        with torch.no_grad():
            feats = self.model.get_text_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()

    def encode_images(self, images):
        inputs = self.processor(images=images, return_tensors="pt").to(self.device)
        with torch.no_grad():
            feats = self.model.get_image_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()

    def add_texts(self, texts):
        for text in texts:
            self.text_data.append(text)
            emb = self.encode_texts([text])[0]
            self.embeddings.append(emb)
            self.metadata.append(("text", len(self.text_data) - 1))

    def add_images(self, images, captions=None):
        if self.captioner:
            captions = [caption if caption else self.captioner.caption(img) for img, caption in zip(images, captions or [None] * len(images))]
        else:
            captions = captions or [None] * len(images)

        for img, caption in zip(images, captions):
            if img is None:
                continue
            self.image_data.append(img)
            img_idx = len(self.image_data) - 1

            img_emb = self.encode_images([img])[0]
            self.embeddings.append(img_emb)
            self.metadata.append(("image", img_idx))

            if caption:
                self.image_captions.append(caption)
                cap_emb = self.encode_texts([caption])[0]
                self.embeddings.append(cap_emb)
                self.metadata.append(("caption", img_idx))
            else:
                self.image_captions.append(None)

    def build_index(self):
        if self.embeddings:
            all_embs = np.vstack(self.embeddings).astype("float32")
            self.index = faiss.IndexFlatIP(all_embs.shape[1])
            self.index.add(all_embs)

    def encode_query(self, query_text=None, query_image=None):
        if query_text and query_image:
            text_emb = self.encode_texts([query_text])
            image_emb = self.encode_images([query_image])
            combined = text_emb + image_emb
            return combined / np.linalg.norm(combined, axis=1, keepdims=True)
        elif query_text:
            return self.encode_texts([query_text])
        elif query_image:
            return self.encode_images([query_image])
        else:
            raise ValueError("Provide text and/or image as query.")

    def retrieve(self, query_text=None, query_image=None, top_k=10, max_kl_threshold=0.4):
        query_emb = self.encode_query(query_text, query_image)
        scores, indices = self.index.search(query_emb, top_k)
        context_texts, context_images = [], []
        for i in indices[0]:
            doc_type, doc_idx = self.metadata[i]
            if doc_type == "text":
                doc_emb = self.encode_texts([self.text_data[doc_idx]])[0]
            elif doc_type == "image":
                doc_emb = self.encode_images([self.image_data[doc_idx]])[0]
            elif doc_type == "caption":
                caption_text = self.image_captions[doc_idx]
                if caption_text:
                    doc_emb = self.encode_texts([caption_text])[0]
                else:
                    continue
            else:
                continue

            divergence = js_divergence(query_emb[0], doc_emb)
            if divergence <= max_kl_threshold:
                if doc_type == "text":
                    context_texts.append(self.text_data[doc_idx])
                elif doc_type == "caption":
                    context_texts.append(self.image_captions[doc_idx])
                    context_images.append(self.image_data[doc_idx])
                elif doc_type == "image":
                    context_texts.append("[Image]")
                    context_images.append(self.image_data[doc_idx])
        return context_texts, context_images

class MultimodalGenerator:
    def __init__(self, model_name="llava-hf/llava-1.5-7b-hf"):
        self.model_name = model_name
        self.processor = AutoProcessor.from_pretrained(self.model_name)
        self.model = LlavaForConditionalGeneration.from_pretrained(
            self.model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto"
        )

    def generate(self, query, query_image=None, context_texts=[], context_images=None):
        img_list = []
        if query_image:
            question_string = f"<image>\n{query}"
        else:
            question_string = query

        context = "\n".join(context_texts)
        if context_images:
            img_place_holder = "<image>" * len(context_images)
            context_string = f"{img_place_holder}\n{context}"
            [img_list.append(im_ctx) for im_ctx in context_images]
        else:
            context_string = context

        if query_image:
            img_list.append(query_image)

        prompt_text = f"Context:{context_string}\n\nQuestion: {question_string}\n"
        prompt = f"USER: {prompt_text}\nASSISTANT:"

        if img_list:
            inputs = self.processor(img_list, prompt, padding=True, return_tensors="pt").to("cuda")
        else:
            inputs = self.processor(prompt, padding=True, return_tensors="pt").to("cuda")
        output = self.model.generate(**inputs, max_new_tokens=200)
        generated_text = self.processor.batch_decode(output, skip_special_tokens=True)

        return generated_text[0].split("ASSISTANT:")[-1]

class MultimodalRAG:
    def __init__(self):
        self.retriever = SigLIPRetriever()
        # self.retriever.load_auto_captioner()
        self.generator = MultimodalGenerator()

    def add_texts(self, texts):
        self.retriever.add_texts(texts)

    def add_images(self, images, captions=None):
        self.retriever.add_images(images, captions)

    def build_index(self):
        self.retriever.build_index()

    def query(self, query_text=None, query_image=None, top_k=10, max_kl_threshold=0.5):
        ctx_texts, ctx_images = self.retriever.retrieve(
            query_text=query_text,
            query_image=query_image,
            top_k=top_k,
            max_kl_threshold=max_kl_threshold
        )
        return self.generator.generate(
            query=query_text,
            query_image=query_image,
            context_texts=ctx_texts,
            context_images=ctx_images
        )

# ✅ Load Documents & Build RAG
pdf_text_chunks, pdf_images = [], []
pdf_files = ["/content/drive/MyDrive/RAG_Documents/Full-1.pdf", "/content/drive/MyDrive/RAG_Documents/Full-2.pdf"]
for file_path in pdf_files:
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pdf_text_chunks.extend(text.split("\n"))
            for image in page.images:
                bbox = (image['x0'], image['top'], image['x1'], image['bottom'])
                pil_img = page.to_image().original.crop(bbox)
                pdf_images.append(pil_img.convert("RGB"))

rag = SigLIPRetriever()
rag.add_texts(pdf_text_chunks)
rag.add_images(pdf_images)
rag.build_index()

# ✅ Load QA pairs
with open("/content/drive/MyDrive/RAG_Documents/engineering_questions.json") as f:
    test_data = json.load(f)

# ✅ Generate predictions WITH RAG
gen = MultimodalGenerator()
rag_preds = [
    gen.generate(query=ex["question"], context_texts=rag.retrieve(query_text=ex["question"])[0]).strip().lower()
    for ex in test_data
]

# ✅ Generate predictions WITHOUT RAG
no_rag_preds = [
    gen.generate(query=ex["question"], context_texts=[]).strip().lower()
    for ex in test_data
]

# ✅ Evaluate BLEU + TopK Accuracy
bleu = evaluate.load("bleu")

rag_bleus = [bleu.compute(predictions=[p], references=[[ex['answer']]])["bleu"] for p, ex in zip(rag_preds, test_data)]
no_rag_bleus = [bleu.compute(predictions=[p], references=[[ex['answer']]])["bleu"] for p, ex in zip(no_rag_preds, test_data)]

def compute_topk_accuracy(preds, refs, k=3):
    correct = 0
    for pred, ref in zip(preds, refs):
        guesses = [g.strip().lower() for g in pred.split(",")][:k]
        if ref["answer"].lower() in guesses:
            correct += 1
    return correct / len(refs)

# ✅ Report
print("\nRAG Evaluation Results")
for i, (ex, p, s) in enumerate(zip(test_data, rag_preds, rag_bleus)):
    print(f"Q{i+1}: {ex['question']}\nTarget: {ex['answer']}\nPred:   {p}\nBLEU:   {s:.2f}\n")
print(f"✅ RAG Avg BLEU: {np.mean(rag_bleus)*100:.2f}%")
print(f"✅ RAG Top-3 Acc: {compute_topk_accuracy(rag_preds, test_data, k=3)*100:.2f}%")

print("\nNo-RAG Evaluation Results")
for i, (ex, p, s) in enumerate(zip(test_data, no_rag_preds, no_rag_bleus)):
    print(f"Q{i+1}: {ex['question']}\nTarget: {ex['answer']}\nPred:   {p}\nBLEU:   {s:.2f}\n")
print(f"✅ No-RAG Avg BLEU: {np.mean(no_rag_bleus)*100:.2f}%")
print(f"✅ No-RAG Top-3 Acc: {compute_topk_accuracy(no_rag_preds, test_data, k=3)*100:.2f}%")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

You may have used the wrong order for inputs. `images` should be passed before `text`. The `images` and `text` inputs will be swapped. This behavior will be deprecated in transformers v4.47.



RAG Evaluation Results
Q1: What are the three broad categories of mechanical engineering sub-disciplines?
Target: The three broad categories of mechanical engineering sub-disciplines are System Design, Energy, and Biomechanical Engineering.
Pred:   the three broad categories of mechanical engineering sub-disciplines are:

1. thermal and fluid systems
2. mechanical design and manufacturing
3. controls and robotics
BLEU:   0.27

Q2: What are the fundamental mechanics courses in mechanical engineering?
Target: The fundamental mechanics courses are Statics, Dynamics, Thermodynamics, Solid Mechanics/Materials, and Fluid Mechanics.
Pred:   the fundamental mechanics courses in mechanical engineering typically include:

1. statics: the study of forces and their effects on objects at rest.
2. dynamics: the study of forces and their effects on moving objects.
3. thermodynamics: the study of the relationships between heat, work, and energy.
4. mechanics of materials: the study of the behavior of

In [ ]:
# ✅ INSTALL DEPENDENCIES (Colab-compatible)
!pip install -q datasets transformers accelerate bitsandbytes faiss-cpu xformers peft torchvision pdfplumber sentence-transformers evaluate

# ✅ IMPORTS
from datasets import load_dataset
from PIL import Image
import torch
import numpy as np
import faiss
import random
from collections import Counter
import os
import json
import torch.nn.functional as F
import pdfplumber
import evaluate
from transformers import (
    BitsAndBytesConfig, AutoProcessor, LlavaForConditionalGeneration,
    CLIPProcessor, CLIPModel, BlipProcessor, BlipForConditionalGeneration
)
from sentence_transformers import SentenceTransformer

random.seed(42)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

class LightweightBLIPCaptioner:
    def __init__(self, model_name="Salesforce/blip-image-captioning-base", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = BlipProcessor.from_pretrained(model_name)
        self.model = BlipForConditionalGeneration.from_pretrained(model_name).to(self.device)

    def caption(self, image):
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=64)
        return self.processor.batch_decode(out, skip_special_tokens=True)[0]

# ✅ CLIP Text-Image Retriever
class CLIPTextImageRetriever:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model_id = "openai/clip-vit-large-patch14"
        self.text_model_id = "sentence-transformers/clip-ViT-L-14"
        self.clip_processor = CLIPProcessor.from_pretrained(self.clip_model_id)
        self.clip_model = CLIPModel.from_pretrained(self.clip_model_id).to(self.device)
        self.st_model = SentenceTransformer(self.text_model_id).to(self.device)
        self.text_data, self.image_data, self.image_captions, self.embeddings, self.metadata = [], [], [], [], []
        self.index = None
        self.captioner = None

    def encode_texts(self, texts):
        return self.st_model.encode(texts, normalize_embeddings=True)

    def encode_images(self, images):
        inputs = self.clip_processor(images=images, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            feats = self.clip_model.get_image_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()

    def add_texts(self, texts):
        for text in texts:
            self.text_data.append(text)
            emb = self.encode_texts([text])[0]
            self.embeddings.append(emb)
            self.metadata.append(("text", len(self.text_data) - 1))

    def add_images(self, images, captions=None):
        if self.captioner:
            captions = [caption if caption else self.captioner.caption(img) for img, caption in zip(images, captions or [None] * len(images))]
        else:
            captions = captions or [None] * len(images)

        for img, caption in zip(images, captions):
            if img is None:
                continue
            self.image_data.append(img)
            img_idx = len(self.image_data) - 1

            img_emb = self.encode_images([img])[0]
            self.embeddings.append(img_emb)
            self.metadata.append(("image", img_idx))

            if caption:
                self.image_captions.append(caption)
                cap_emb = self.encode_texts([caption])[0]
                self.embeddings.append(cap_emb)
                self.metadata.append(("caption", img_idx))
            else:
                self.image_captions.append(None)

    def build_index(self):
        if self.embeddings:
            all_embs = np.vstack(self.embeddings).astype("float32")
            self.index = faiss.IndexFlatIP(all_embs.shape[1])
            self.index.add(all_embs)

    def encode_query(self, query_text=None, query_image=None):
        if query_text and query_image:
            text_emb = self.encode_texts([query_text])
            image_emb = self.encode_images([query_image])
            combined = text_emb + image_emb
            return combined / np.linalg.norm(combined, axis=1, keepdims=True)
        elif query_text:
            return self.encode_texts([query_text])
        elif query_image:
            return self.encode_images([query_image])
        else:
            raise ValueError("Provide text and/or image as query.")

    def retrieve(self, query_text=None, query_image=None, top_k=10):
        q_emb = self.encode_query(query_text, query_image)
        scores, indices = self.index.search(q_emb, top_k)
        ctx_texts, ctx_images = [], []
        for i in indices[0]:
            doc_type, doc_idx = self.metadata[i]
            if doc_type == "text":
                ctx_texts.append(self.text_data[doc_idx])
            elif doc_type == "caption":
                ctx_texts.append(self.image_captions[doc_idx])
                ctx_images.append(self.image_data[doc_idx])
            else:
                ctx_texts.append("[Image]")
                ctx_images.append(self.image_data[doc_idx])
        return ctx_texts, ctx_images

# ✅ Generator
class MultimodalGenerator:
    def __init__(self, model_name="llava-hf/llava-1.5-7b-hf"):
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = LlavaForConditionalGeneration.from_pretrained(
            model_name, torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
             device_map="auto")

    def generate(self, query, query_image=None, context_texts=[], context_images=[]):
        img_list = []
        if query_image:
            question_string = f"<image>\n{query}"
        else:
            question_string = query

        context = "\n".join(context_texts)
        if context_images:
            img_place_holder = "<image>" * len(context_images)
            context_string = f"{img_place_holder}\n{context}"
            [img_list.append(im_ctx) for im_ctx in context_images]
        else:
            context_string = context

        if query_image:
            img_list.append(query_image)

        prompt_text = f"Context:{context_string}\n\nQuestion: {question_string}\n"
        prompt = f"USER: {prompt_text}\nASSISTANT:"

        if img_list:
            inputs = self.processor(img_list, prompt, padding=True, return_tensors="pt").to("cuda")
        else:
            inputs = self.processor(prompt, padding=True, return_tensors="pt").to("cuda")
        output = self.model.generate(**inputs, max_new_tokens=200)
        generated_text = self.processor.batch_decode(output, skip_special_tokens=True)

        return generated_text[0].split("ASSISTANT:")[-1]

# ✅ CLIP-based RAG Wrapper
class CLIPMultimodalRAG:
    def __init__(self):
        self.retriever = CLIPTextImageRetriever()
        self.generator = MultimodalGenerator()

    def add_texts(self, texts): self.retriever.add_texts(texts)
    def add_images(self, images, captions=None): self.retriever.add_images(images, captions)
    def build_index(self): self.retriever.build_index()

    def query(self, query_text=None, query_image=None, top_k=10):
        ctx_texts, ctx_images = self.retriever.retrieve(query_text, query_image, top_k)
        return self.generator.generate(query=query_text, query_image=query_image, context_texts=ctx_texts, context_images=ctx_images)


# ✅ Load Documents & Build RAG
pdf_text_chunks, pdf_images = [], []
pdf_files = ["/content/drive/MyDrive/RAG_Documents/Full-1.pdf", "/content/drive/MyDrive/RAG_Documents/Full-2.pdf"]
for file_path in pdf_files:
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pdf_text_chunks.extend(text.split("\n"))
            for image in page.images:
                bbox = (image['x0'], image['top'], image['x1'], image['bottom'])
                pil_img = page.to_image().original.crop(bbox)
                pdf_images.append(pil_img.convert("RGB"))

rag = CLIPTextImageRetriever()
rag.add_texts(pdf_text_chunks)
rag.add_images(pdf_images)
rag.build_index()

# ✅ Load QA pairs
with open("/content/drive/MyDrive/RAG_Documents/engineering_questions.json") as f:
    test_data = json.load(f)

# ✅ Generate predictions WITH RAG
gen = MultimodalGenerator()
rag_preds = [
    gen.generate(query=ex["question"], context_texts=rag.retrieve(query_text=ex["question"])[0]).strip().lower()
    for ex in test_data
]

# ✅ Generate predictions WITHOUT RAG
no_rag_preds = [
    gen.generate(query=ex["question"], context_texts=[]).strip().lower()
    for ex in test_data
]

# ✅ Evaluate BLEU + TopK Accuracy
bleu = evaluate.load("bleu")

rag_bleus = [bleu.compute(predictions=[p], references=[[ex['answer']]])["bleu"] for p, ex in zip(rag_preds, test_data)]
no_rag_bleus = [bleu.compute(predictions=[p], references=[[ex['answer']]])["bleu"] for p, ex in zip(no_rag_preds, test_data)]

def compute_topk_accuracy(preds, refs, k=3):
    correct = 0
    for pred, ref in zip(preds, refs):
        guesses = [g.strip().lower() for g in pred.split(",")][:k]
        if ref["answer"].lower() in guesses:
            correct += 1
    return correct / len(refs)

# ✅ Report
print("\nRAG Evaluation Results")
for i, (ex, p, s) in enumerate(zip(test_data, rag_preds, rag_bleus)):
    print(f"Q{i+1}: {ex['question']}\nTarget: {ex['answer']}\nPred:   {p}\nBLEU:   {s:.2f}\n")
print(f"✅ RAG Avg BLEU: {np.mean(rag_bleus)*100:.2f}%")
print(f"✅ RAG Top-3 Acc: {compute_topk_accuracy(rag_preds, test_data, k=3)*100:.2f}%")

print("\nNo-RAG Evaluation Results")
for i, (ex, p, s) in enumerate(zip(test_data, no_rag_preds, no_rag_bleus)):
    print(f"Q{i+1}: {ex['question']}\nTarget: {ex['answer']}\nPred:   {p}\nBLEU:   {s:.2f}\n")
print(f"✅ No-RAG Avg BLEU: {np.mean(no_rag_bleus)*100:.2f}%")
print(f"✅ No-RAG Top-3 Acc: {compute_topk_accuracy(no_rag_preds, test_data, k=3)*100:.2f}%")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]